## Calculating a Typical Meteorological Year

This notebook walks through the process of calculating a [Typical Meteorological Year](https://nsrdb.nrel.gov/data-sets/tmy), an hourly dataset used for applications in energy and building systems modeling. Because this represents average rather than extreme conditions, an TMY dataset is not suited for designing systems to meet the worst-case conditions occurring at a location.

The TMY methodology here mirrors that of the Sandia/NREL TMY3 methodology, and uses historic and projected downscaled (bias corrected?) climate data available through the Cal-Adapt: Analytics Engine catalog. 

Please note, the Analytics Engine also has an *Average Meteorological Year* functionality. The methods shown throughout this notebook will soon replace the underlying backend `climakitae` code for the AMY in order to better address our user needs, i.e., we are working to replace the AMY with the TMY methods. We provide this walkthrough to demonstrate confidence in the "AMY to TMY" conversion process for our users in the meantime. 


**Notes for dev**
1. Check data sizes
2. Point location rather than area
    - start with station based, but think about flexible for any location
3. avoid hardcoding dates for solar zenith angle

### Step 0: Set-up

Import the [climakitae](https://github.com/cal-adapt/climakitae) library and other dependencies.

In [ ]:
import climakitae as ck
import pandas as pd
import xarray as xr
import numpy as np
from statsmodels.distributions.empirical_distribution import ECDF
import matplotlib.pyplot as plt
import pkg_resources
import holoviews
from datetime import date

# to calculate direct normal irradiance
!pip install pvlib
from pvlib import irradiance, solarposition
import pvlib

from climakitae.utils import get_closest_gridcell
from climakitae.derive_variables import _compute_dewpointtemp
from climakitae.unit_conversions import _convert_units

To use climakitae, load a new application:

In [ ]:
app = ck.Application()

### Step 1: Grab and process all required input data

The TMY3 method selects a "typical" month based on ten daily variables: max, min, and mean air and dew point temperatures, max and mean wind speed, global irradiance and direct irradiance ([NREL TMY3](https://www.nrel.gov/docs/fy08osti/43156.pdf)). Some of these variables are already available in the Analytics Engine data catalog, and some we will need to calculate ourselves.  

#### Step 1a: Variables in catalog
When selecting data, there are several considerations to be aware of. The recommended minimum input of data is 15-20 years worth of daily data, and includes leap days. First, we will pre-load some data options to ensure that the same constraints are applied to every variable we retrieve from the catalog and calculate. For example, we will process all data for *Alameda County* at *45 km* over the 2010-2020 period. For data post-2014, we will utilize SSP 3-7.0, although scenario selection in the near-future is relatively independent. If calculating a TMY for the far-future, **carefully consider which scenario SSP to include**, as there will be **significant** differences present. 

In [ ]:
# default selections applicable to all variables selected
app.selections.data_type = "Gridded"
app.selections.area_average = "Yes"
app.selections.scenario_historical = ["Historical Climate"]
app.selections.scenario_ssp = ["SSP 3-7.0 -- Business as Usual"] # Important based on time period considered!
app.selections.time_slice = (1990, 2020) # Using 10 year as baseline to start, and cut down on data read in testing
app.selections.timescale = "daily"
app.selections.resolution = "45 km"
app.selections.area_subset = "CA counties"
app.selections.cached_area = "Alameda County"

Retrieve min, max, and mean air temperature: 

In [ ]:
# max air temp
app.selections.variable = "Daily maximum air temperature at 2m"
app.selections.unit = "degC"
max_airtemp_data = app.retrieve()

# min air temp
app.selections.variable = "Daily minimum air temperature at 2m"
app.selections.unit = "degC"
min_airtemp_data = app.retrieve()

# mean air temperature
app.selections.variable = "Air Temperature at 2m" 
app.selections.unit = "degC"
mean_airtemp_data = app.retrieve()

Retrieve max and mean wind speed:

In [ ]:
# max wind speed
app.selections.variable = "Maximum wind speed at 10m"
app.selections.unit = "m s-1"
max_windspd_data = app.retrieve()

# mean wind speed
app.selections.variable = "Mean wind speed at 10m"
app.selections.unit = "m s-1"
mean_windspd_data = app.retrieve()

#### Step 1b: Variables that need to be calculated
Next, we will need to retrieve **hourly** data from the catalog to calculate the following variables, as they are not natively within the Analytics Engine data catalog. 
- Max, min, and mean dew point temperature
- Direct normal irradiance
- Global horizontal irradiance

In [ ]:
# switch to hourly timescale
app.selections.timescale = "hourly"

Next, calculate max, min, and mean dew point temperature:

In [ ]:
# dew point temperature
app.selections.variable = "Dew point temperature"
app.selections.unit = "degC"
dewpt_data = app.retrieve()

max_dptemp_data = dewpt_data.resample(time="1D").max() # daily max dewpoint temp
max_dptemp_data.name = "Daily max dewpoint temperature" # rename for clarity

min_dptemp_data = dewpt_data.resample(time="1D").min() # daily min dewpoint temp
min_dptemp_data.name = "Daily min dewpoint temperature" # rename for clarity

mean_dptemp_data = dewpt_data.resample(time="1D").mean() # daily mean dewpoint temp
mean_dptemp_data.name = "Daily mean dewpoint temperature" # rename for clarity

Lastly, we will calculate **direct normal irradiance**. As direct normal irradiance is not presently in the Analytics Engine catalog (coming soon!), we will use the DIRINTS method (Perez et al. 1992 -- looking for link) to calculate, which uses the global horizontal irradiance (which we need as our last variable!) and the solar zenith angle. We'll process and grab global horizontal irradiance in this process too, which is provided in the Analytics Engine catalog. Because solar zenith angle requires knowing the latitude and longitude of the region of interest, we will retrieve the non-area averaged global irradiance data, and then area average it a step later. 

**notes**
- *Perez, R., P. Ineichen, E. Maxwell, R. Seals and A. Zelenka, (1992). “Dynamic Global-to-Direct Irradiance Conversion Models”. ASHRAE Transactions-Research Series, pp. 354-369*
- *Different methods: https://assessingsolar.org/notebooks/decomposition_models.html*
    - Victoria confirmed that ERBS is not ideal

In [ ]:
# global irradiance
app.selections.variable = "Instantaneous downwelling shortwave flux at bottom"
app.selections.unit = "W m-2"
app.selections.area_average = "No"
global_irradiance_data = app.retrieve()
global_irradiance_data.name = "Global horizontal irradiance" # rename for clarity

In [ ]:
# grab the "average" lat and lon of a region -- need to think on this further, computationally expensive to do per grid cell, but possible
avg_lat = global_irradiance_data.lat.mean()
avg_lon = global_irradiance_data.lon.mean()

Next, we'll process the global horizontal irradiance data by taking an area average, and resampling the data back to daily temporal resolution, which is what we need for the TMY calculation.

In [ ]:
# area average GHI data
weights = np.cos(np.deg2rad(global_irradiance_data.lat))
ghi_area = global_irradiance_data.weighted(weights).mean("x").mean("y").squeeze() # area-average

ghi_daily = ghi_area.resample(time="1D").mean()

Now we need to calculate the solar zenith angle. 

In [ ]:
# calculate solar position, including solar zenith angle
solpos = solarposition.get_solarposition(time = pd.date_range('1990-01-01', '2020-12-31', freq='D'), # need to avoid hardcoding dates here... in progress
                                        latitude = avg_lat.values,
                                        longitude = avg_lon.values)
solpos

Next, we will set-up a helper function `dni_by_sim` to calculate direct normal irradiance. 

In [ ]:
def dni_dirint(data, solpos):
    """Calculates the direct normal irradiance per simulation, with DIRINT method"""
    df = pd.DataFrame()  
    for sim in range(0, 4):        
        df_by_sim = data.squeeze().isel(simulation=sim).to_dataframe()
        dirint_by_sim = pvlib.irradiance.dirint(ghi=df_by_sim['Global horizontal irradiance'],
                                                solar_zenith=solpos['apparent_zenith'],
                                                # temp_dew=mean_dptemp_data.squeeze().isel(simulation=sim).to_dataframe(),
                                                times=solpos.index)
        
        df[df_by_sim.simulation.values[0]] = dirint_by_sim      
    return df

Calculate direct normal irradiance and check out the values next:

In [ ]:
direct_irradiance = dni_dirint(ghi_daily, solpos)
direct_irradiance

In [ ]:
# convert back to xr.Dataset, dims=sims

#### Step 1d: Load all variables
Now that we have all of our data retrieved and calculated, it is time to actually load the data into memory. Previously, xarray has lazily loaded the data, but we will actually grab it now. 

In [ ]:
# load all indices in
max_airtemp_data = app.load(max_airtemp_data.squeeze())
min_airtemp_data = app.load(min_airtemp_data.squeeze())
mean_airtemp_data = app.load(mean_airtemp_data.squeeze())
max_dptemp_data = app.load(max_dptemp_data.squeeze())
min_dptemp_data = app.load(min_dptemp_data.squeeze())
mean_dptemp_data = app.load(mean_dptemp_data.squeeze())
max_windspd_data = app.load(max_windspd_data.squeeze())
mean_windspd_data = app.load(mean_windspd_data.squeeze())
ghi_daily = app.load(ghi_daily)

For convenience, merge all the data together so it is in a single dataset. This will be very useful in the next steps. (tbd)

### Step 2: Calculate cumultative distribution functions

Notes on what a CDF is here: 

*** makes assumptions based on the distribution function here, fyi

#### Step 2a: Calculate long-term climatology CDFs for each index
For each month.....

- so far this is doing this for all data, not monthly selections

In [ ]:
def plot_cdf(data,title):
    for i in range(4):
        cdf = ECDF(data.isel(simulation=i))
        _to_plot = plt.plot(cdf.x, cdf.y)
        plt.title(label=title)
    return _to_plot

In [ ]:
# produce subplots that illustrates each climatological CDF with variable name provided, with month as a toggle selection

In [ ]:
plt.figure(figsize=(6,6))
c,r = 2,2
plt.subplot(c,r,1)
plot_cdf(min_airtemp_data, title="")
plot_cdf(mean_airtemp_data, title="")
plot_cdf(max_airtemp_data, title="Air Temp (min/mean/max)")

plt.subplot(c,r,2)
plot_cdf(min_dptemp_data, title="")
plot_cdf(mean_dptemp_data, title="")
plot_cdf(max_dptemp_data, title="D. temp (min/mean/max")

plt.subplot(c,r,3)
plot_cdf(mean_windspd_data, title="Wind speed (mean/max)")
plot_cdf(max_windspd_data, title="Wind speed (mean/max)")

plt.subplot(c,r,4)
# dni_plots(direct_irradiance, title="")
plot_cdf(ghi_daily, title="Irradiance (Direct/Global)")

#### Step 2b: Calculate CDFs for each index for each year

Calculate CDF for each month and each variable: Gives proportion of values that are less than or equal to a specified value of index/variable
- Must exclude the following months: 
    - El Chichón: May 1982 until December 1984
    - Pinatubo: June 1991 to December 1994
    - Leap year days are excluded in the monthly CDFS

In [ ]:
elchichon = pd.date_range(datetime.date(1982,5,1), datetime.date(1984,12,31),freq='d')
elchichon

In [ ]:
# more convenient if all data was in single ds - merge data
# or write function for all vars

def remove_volcano(data):
    """Excludes specific months from CDF calculation due to volcanic eruptions"""
    print(date.time[0], date.time[-1])
    
    # el chichón: 5/1/1982 - 12/31/1984
    elchichon = pd.date_range(datetime.date(1982,5,1), datetime.date(1984,12,31),freq='d')
        
    # need to check start date of data - depending on selection, these dates may be within time, partially, or not at all
    if data.time[0].values not in elchichon and data.time[-1].values not in elchichon: # no data within time record
        continue # skip overall
        
    elif data.time[0].values not in elchichon and data.time[-1].values in elchichon: # data has a partial date coverage at end
        continue # come back to this
    
    elif data.time[0].values in elchichon and data.time[-1].values not in elchichon: # data has partial date coverage at start
        continue # come back to this
        
    else: # should be start and end are in date range
        data.drop_sel(time=elchichon)
    

    # pinatubo: 6/1/1991 - 12/31/1994
    pinatubo = pd.date_range(datetime.date(1991,6,1), datetime.date(1994,12,31),freq='d')

    # same process here
    # data.where(~pinatubo, drop=True)
    data.drop_sel(time=pinatubo)
    
    return data

def remove_leapdays(data):
    """Drops Feb 29th in leap years"""
    leap_m = np.logical_and(data.time.dt.is_leap_year, data.time.dt.dayofyear==60) # identifies leap years
    return data.where(~leap_m, drop=True) # drops feb 29 only if leap year

In [ ]:
max_t = remove_volcano(max_airtemp_data)
# max_t = remove_leapdays(max_t)
max_t
# remove_leapdays(min_airtemp_data)

In [ ]:
max_t.hvplot.line(x='time')

### Step 3: Compare climatology CDF to monthly CDF for each variable

#### Step 3a: Calculate the Finkelstein-Schafer statistic 
REF HERE
Absolute difference between long-term climatology CDF and candidate CDF divided by number of days per month

In [ ]:
def fs_statistic(cdf_climatology, cdf_month):
    """
    Calculates the Finkelstein-Schafer statistic:
    Absolute difference between long-term climatology and candidate CDF, divided by number of days in month
    """
    len_month = number_of_days_in_month
    da_fs = (cdf_climatology - cdf_month).abs().mean()
    return da_fs / len_month


# Absolute difference between long-term climatology CDF and candidate CDF divided by number of days in month
# Finkelstein-Schafer statistic (difference between long term  CDF and year CDF

#### Step 3b: Weight the F-S statistic
WS = Weight * F-S value

- Max/min air temp, max/min dry bulb temp, max/mean wind speed 1/20 each
- Mean air temp, mean dew point 2/20 each
- Global irradiance, direct irradiance 5/20 each

In [ ]:
def weighted_fs(da_fs):
    """Weights the F-S statistics based on TMY3 methodology"""
    
    
    return da_fs_weighted

#### Step 3c: Select candidate months for consideration
Once weighted, select 5 candidate months that have lowest weighted sums

In [ ]:
# for each month, rank all available years, lowest to highest
# select first five as "candidates"

#### Step 3d: Rank candidate months for selection
Rank with respect to closeness of month to long-term mean and median

"Typical" month selected by choosing among 5 months with lowest WS values the month with the smallest deviation from the long-term CDF

In [ ]:
# using 5 candidate months
# identify which month is closest to climatology

### Step 4: Consider persistence of weather patterns

#### Step 4a: Monthly mean and median, persistence of wx patterns
- Mean daily dry bulb temp
- Global horizontal radiation
- frequency and run length above 67th percentile/below 33rd percentile

Exclude month with longest run, month with most runs, month with zero runs

***Noting that https://github.com/TerriaJS/aremi-tmy/blob/master/sandia/tmy.py#L118 skips this step***

### Step 5: Concatenate months together

Months concatenated together and discontinuities at month interface are smoothed for 6 hours on either side using curve-fitting techniques

In [ ]:
# provide a list/table/output of which month comes from which year

### Step 6: Generate the TMY data outputs

Generally, the following is outputted using the TMY months:
- Date & time (UTC)
- T2m [°C] - Dry bulb (air) temperature.
- RH [%] - Relative Humidity.
- G(h) [W/m2] - Global horizontal irradiance.
- Gb(n) [W/m2] - Direct (beam) irradiance -- ***currently not available to calculate in AE -- but can be calculated here and added in as derived variable?***
- Gd(h) [W/m2] - Diffuse horizontal irradiance.
- IR(h) [W/m2] - Infrared radiation downwards -- ***Instantaneous downwelling longwave flux at bottom -- check that this is what we want here***
- WS10m [m/s] - Windspeed.
- WD10m [°] - Wind direction. -- ***currently not available to calculate in AE***
- SP [Pa] - Surface (air) pressure.

In [ ]:
# step 1 - using TMY selction of months
# step 2 - grab available variables at hourly timescales as dataarrays
# step 3 - concat variable dataarrays into pandas df
# step 4 - visualize (head) table
# step 5 - export table ==> this is the "TMY data file"